# 06 · FABLE crop mapping

Review all 56 canonical FABLE products against the exact 12 direct LPJmL crops. No unsupported season splits, proxy selection or route values are generated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

REFERENCE=DATA/'crop'/'evidence'/'02_FABLE_56_product_reference.csv'
OUTPUT=RUNS/'crop-mapping'/'56-product-direct-support.csv'
FABLE_TO_LPJML={'cassava':'cas','corn':'mai','millet':'mil','groundnut':'nut',
 'peas':'pea','rapeseed':'rap','rice':'ric','sugarbeet':'sgb',
 'soyabean':'soy','sugarcane':'sug','sunflower':'sun','wheat':'whe'}

In [ ]:
if not REFERENCE.is_file():raise FileNotFoundError(REFERENCE)
product_ref=pd.read_csv(REFERENCE)
if 'fable_product' not in product_ref:raise ValueError('Canonical product column missing')
if len(product_ref)!=56 or product_ref.fable_product.nunique()!=56:
    raise ValueError('Expected 56 unique FABLE products')
product_ref['fable_product_norm']=product_ref.fable_product.astype(str).str.strip().str.lower()
product_ref['lpjml_crop_code']=product_ref.fable_product_norm.map(FABLE_TO_LPJML)
product_ref['support_class']=np.where(product_ref.lpjml_crop_code.notna(),
 'DIRECT_LPJML','REQUIRES_ACCEPTED_PROXY_OR_OTHER_ROUTE_EVIDENCE')
assert product_ref.support_class.eq('DIRECT_LPJML').sum()==12
print(product_ref[['fable_product','lpjml_crop_code','support_class']].to_string(index=False))

In [ ]:
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists():raise FileExistsError(OUTPUT)
product_ref.to_csv(OUTPUT,index=False)